# Task B: Inheritance

You have been given a ready‑made binary classification pipeline. The attached baseline reports `ROC‑AUC ≈ 0.97` on its validation set.

Each row corresponds to an event involving a certain entity at the time specified by event_time. In production, the model is applied to later events. Among the future data, there are both known entities and groups, as well as those that have not been observed before.

The numerical features f_00–f_23 were obtained using the existing feature pipeline. Its documentation is incomplete, so the correctness of individual features and the original validation protocol cannot be guaranteed.

Your goal is to achieve the highest possible score on the hidden future period. You are allowed to change:
* the validation, 
* the set of features, 
* preprocessing, 
* the model, 
* the training method.

## files:
* `train.csv             —` features and target;
* `test.csv              —` a later period without target;
* `starter.py            —` the original baseline;
* `sample_submission.csv —` a sample submission;
* `requirements.txt      —` minimal environment.

## main columns:
* `row_id —` row identifier;
* `event_time —` event time in arbitrary seconds;
* `entity_id, group_id —` recurring categorical identifiers;
* `cat_00–cat_04 —` categorical features;
* `f_00–f_23 —` anonymous numerical features;
* `target —` binary target variable, available only in `train`.

**NB!:** Rows in `train.csv` are shuffled. Rows in `test.csv` belong to a period subsequent to `train`.

## submission format
You must submit only a CSV file:

row_id,prediction

10000001,0.031

10000002,0.772

## requirements:

1) exactly one row for each `row_id` from `test.csv`;
2) columns must be named `row_id` and `prediction`;
3) prediction must be a finite number between `0` and `1`;
4) extra, missing, or duplicate `row_id` values are not allowed;
5) no text answers or code need to be submitted.

## metrics
The hidden test is divided into three consecutive time blocks. Let `A1`, `A2`, `A3` denote the `ROC‑AUC` on these blocks.

A cold‑start subset is defined separately: rows where `entity_id` or `group_id` did not appear in train. Let `Acold` denote the `ROC‑AUC` on this subset.

### robust quality $R$ is calculated first:

$$
R \;=\; 0.50 \cdot mean(A_1, A_2, A_3) \;+\; 0.25 \cdot A_{cold} \;+\; 0.25 \cdot min(A_1, A_2, A_3, A_{cold})
$$

### final score:

$$
Score \;=\; 100 \;\cdot\; clip\left(\,\frac{(R - 0.55)}{0.20}, \, 0, \, 1\right)
$$

### benchmarks:
* `0 points  :` R ≤ 0.55;
* `50 points :` R = 0.65;
* `75 points :` R = 0.70;
* `100 points:` R ≥ 0.75.

The higher the score, the better. The best submission counts.

In [40]:
import sys
from pathlib import Path

# Добавляем корневую папку проекта в Python path
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from datetime import datetime
start_run = datetime.now()

In [41]:
import pandas as pd

from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.impute import SimpleImputer

from src.src_B.denoising_for_rare_class import RobustTailFeatures, DistributionFeatures

# Deliberately naive starter baseline

## read datasets

In [42]:
train = pd.read_csv("../data/data_B/train.csv")
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 140035 entries, 0 to 140034
Data columns (total 34 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   row_id      140035 non-null  int64  
 1   event_time  140035 non-null  int64  
 2   entity_id   140035 non-null  int64  
 3   group_id    140035 non-null  int64  
 4   cat_00      140035 non-null  int64  
 5   cat_01      140035 non-null  int64  
 6   cat_02      140035 non-null  int64  
 7   cat_03      140035 non-null  int64  
 8   cat_04      140035 non-null  int64  
 9   f_00        140035 non-null  float64
 10  f_01        140035 non-null  float64
 11  f_02        140035 non-null  float64
 12  f_03        140035 non-null  float64
 13  f_04        140035 non-null  float64
 14  f_05        140035 non-null  float64
 15  f_06        140035 non-null  float64
 16  f_07        140035 non-null  float64
 17  f_08        140035 non-null  float64
 18  f_09        140035 non-null  float64
 19  f_10        1

In [43]:
test = pd.read_csv("../data/data_B/test.csv")
test.info()

<class 'pandas.DataFrame'>
RangeIndex: 63270 entries, 0 to 63269
Data columns (total 33 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   row_id      63270 non-null  int64  
 1   event_time  63270 non-null  int64  
 2   entity_id   63270 non-null  int64  
 3   group_id    63270 non-null  int64  
 4   cat_00      63270 non-null  int64  
 5   cat_01      63270 non-null  int64  
 6   cat_02      63270 non-null  int64  
 7   cat_03      63270 non-null  int64  
 8   cat_04      63270 non-null  int64  
 9   f_00        63270 non-null  float64
 10  f_01        63270 non-null  float64
 11  f_02        63270 non-null  float64
 12  f_03        63270 non-null  float64
 13  f_04        63270 non-null  float64
 14  f_05        63270 non-null  float64
 15  f_06        63270 non-null  float64
 16  f_07        63270 non-null  float64
 17  f_08        63270 non-null  float64
 18  f_09        63270 non-null  float64
 19  f_10        63270 non-null  float64


### look at the target statistics

In [44]:
train.target.value_counts(dropna=False, normalize=True) * 100

target
0    97.444924
1     2.555076
Name: proportion, dtype: float64

### check if row_id is unique

In [45]:
train.row_id.value_counts(dropna=False)

row_id
10082466    1
10022591    1
10133054    1
10092682    1
10070012    1
           ..
10033930    1
10045529    1
10120962    1
10080876    1
10118087    1
Name: count, Length: 140035, dtype: int64

In [46]:
test.row_id.value_counts(dropna=False)

row_id
10140035    1
10140036    1
10140037    1
10140038    1
10140039    1
           ..
10203300    1
10203301    1
10203302    1
10203303    1
10203304    1
Name: count, Length: 63270, dtype: int64

### check if time separation is valid

In [47]:
train.event_time.max() < test.event_time.min()

np.True_

## select and detect fields type

In [48]:
features = [c for c in test.columns if c != "row_id"]
#features

In [49]:
cat_features = ["entity_id", "group_id"] + [c for c in features if c.startswith("cat_")]
cat_features

['entity_id', 'group_id', 'cat_00', 'cat_01', 'cat_02', 'cat_03', 'cat_04']

### changing categorical columns to string-type

In [50]:
for frame in (train, test):
    for col in cat_features:
        frame[col] = frame[col].astype(str)

train.info()

<class 'pandas.DataFrame'>
RangeIndex: 140035 entries, 0 to 140034
Data columns (total 34 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   row_id      140035 non-null  int64  
 1   event_time  140035 non-null  int64  
 2   entity_id   140035 non-null  str    
 3   group_id    140035 non-null  str    
 4   cat_00      140035 non-null  str    
 5   cat_01      140035 non-null  str    
 6   cat_02      140035 non-null  str    
 7   cat_03      140035 non-null  str    
 8   cat_04      140035 non-null  str    
 9   f_00        140035 non-null  float64
 10  f_01        140035 non-null  float64
 11  f_02        140035 non-null  float64
 12  f_03        140035 non-null  float64
 13  f_04        140035 non-null  float64
 14  f_05        140035 non-null  float64
 15  f_06        140035 non-null  float64
 16  f_07        140035 non-null  float64
 17  f_08        140035 non-null  float64
 18  f_09        140035 non-null  float64
 19  f_10        1

## random split of train indicies into cv_train and cv_valid groups
* we know true target values only for train-dataset, thus to estimate metrics on the new data we have to split it too;
* `NB!:` yet baseline uses random split, while not separating past and future data, which augments results on validation data by autocorrelation dependencies found in the training

In [51]:
train_idx, valid_idx = train_test_split(
    range(len(train)),
    test_size=0.25,
    random_state=42,
    stratify=train["target"],
)

## using CatBoost

In [52]:
baseline_model = CatBoostClassifier(
    iterations=140,
    depth=7,
    learning_rate=0.11,
    loss_function="Logloss",
    eval_metric="AUC",
    l2_leaf_reg=6,
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
)

In [53]:
baseline_model.fit(
    train.loc[train_idx, features],
    train.loc[train_idx, "target"],
    cat_features=[features.index(c) for c in cat_features],
)

CatBoostClassifier(allow_writing_files=False, depth=7, eval_metric='AUC', iterations=140, l2_leaf_reg=6, learning_rate=0.11, loss_function='Logloss', random_seed=42, verbose=False)

In [54]:
valid_prediction = baseline_model.predict_proba(train.loc[valid_idx, features])[:, 1]

In [55]:

print(f"Random-split baseline ROC-AUC: {roc_auc_score(train.loc[valid_idx, 'target'], valid_prediction):.6f}")

Random-split baseline ROC-AUC: 0.970494


### retrain model on the whole train dataset

In [56]:

baseline_model.fit(
    train[features],
    train["target"],
    cat_features=[features.index(c) for c in cat_features],
)

CatBoostClassifier(allow_writing_files=False, depth=7, eval_metric='AUC', iterations=140, l2_leaf_reg=6, learning_rate=0.11, loss_function='Logloss', random_seed=42, verbose=False)

### save results for submission

In [57]:
test_prediction = baseline_model.predict_proba(test[features])[:, 1]

pd.DataFrame({"row_id": test["row_id"], "prediction": test_prediction}).to_csv(
                                       "../results/baseline_submission.csv", index=False)

print('FYI: baseline_submission.csv has been saved OK!')


FYI: baseline_submission.csv has been saved OK!


# Improving baseline model follows

## PreProc

### Field: `event_time`

#### add new features for daily and weekly seasonality 
* yearly seasonality is not relevant for the time spread of 9 months only

In [58]:
timestep = 1     # found by better seasonality

train['td'] = pd.to_datetime(train.event_time * timestep, unit='s')
test['td'] = pd.to_datetime(test.event_time   * timestep, unit='s')

In [59]:
train['day_of_week'] = train.td.map(lambda x: x.day_of_week).map(lambda x: 'weekend' if x in [1,2] else 'working')
train['hour_of_day'] = train.td.map(lambda x: 'h_' + str(x.hour))

test['day_of_week'] = test.td.map(lambda x: x.day_of_week).map(lambda x: 'weekend' if x in [1,2] else 'working')
test['hour_of_day'] = test.td.map(lambda x: 'h_' + str(x.hour))

In [60]:
features_seasonal = ['day_of_week','hour_of_day']

### Field: `entity_id`

#### check if there are new `entity_ids` in test

In [61]:
new_entity_ids = []

old_entity_ids = set(train.entity_id.to_list())

for eid in test.entity_id.unique():
    if eid not in old_entity_ids:
        new_entity_ids.append(eid)

print('FYI: there are', len(new_entity_ids), 'new unique entity_ids out of', len(test.entity_id.unique()), 
      'in test data' )

FYI: there are 498 new unique entity_ids out of 3094 in test data


### Field: `group_id` 

#### check if there are new `group_ids` in test

In [62]:
new_group_ids = []

old_group_ids = set(train.group_id.to_list())

for eid in test.group_id.unique():
    if eid not in old_group_ids:
        new_group_ids.append(eid)

print('FYI: there are', len(new_group_ids), 'new unique group_ids out of', len(test.group_id.unique()), 
      'in test data' )

FYI: there are 80 new unique group_ids out of 600 in test data


#### cold start problem found:
* there are about 15% of new entities and groups in test dataset, thus we need a separate model for them

In [63]:
no_id_features = [el for el in features if not el.endswith('_id')]
no_id_cat_features = [el for el in cat_features if not el.endswith('_id')]

### Fields: `cat_00 - cat_04`
no need to preprocess for catboost as we only loose some signal in preprocessing, while catboost shall do it the best way in any case

### Fields: `f_00 - f_23`
* it is given that this data has noise
* specific denoising is needed to keep rare class  
* yet we have not found any useful way

# Train two models:
* for cold start
* for known ids

In [64]:
for lst in [features, no_id_features, cat_features, no_id_cat_features]:
    lst += features_seasonal

## four models - objects construction

In [65]:
cold_start_model = CatBoostClassifier(
    iterations=1000,
    depth=7,
    learning_rate=0.01,
    loss_function="Logloss",
    eval_metric="AUC",
    l2_leaf_reg=6,
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
)

In [66]:
deep_model = CatBoostClassifier(
    iterations=10000,
    depth=7,
    learning_rate=0.01,
    loss_function="Logloss",
    eval_metric="AUC",
    l2_leaf_reg=6,
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
)

In [67]:
import math

math.sqrt(140*1400)

442.7188724235731

In [68]:
no_entity_model = CatBoostClassifier(
    iterations=2000,
    depth=7,
    learning_rate=0.01,
    loss_function="Logloss",
    eval_metric="AUC",
    l2_leaf_reg=6,
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
)

In [69]:
no_group_model = CatBoostClassifier(
    iterations=2000,
    depth=7,
    learning_rate=0.01,
    loss_function="Logloss",
    eval_metric="AUC",
    l2_leaf_reg=6,
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
)

## four models training

In [70]:
cold_start_model.fit(
    train[no_id_features],
    train["target"],
    cat_features=[no_id_features.index(c) for c in no_id_cat_features],
)

CatBoostClassifier(allow_writing_files=False, depth=7, eval_metric='AUC', iterations=1000, l2_leaf_reg=6, learning_rate=0.01, loss_function='Logloss', random_seed=42, verbose=False)

In [71]:
deep_model.fit(
    train[features],
    train["target"],
    cat_features=[features.index(c) for c in cat_features],
)

CatBoostClassifier(allow_writing_files=False, depth=7, eval_metric='AUC', iterations=10000, l2_leaf_reg=6, learning_rate=0.01, loss_function='Logloss', random_seed=42, verbose=False)

In [72]:
no_group_model.fit(
    train[no_id_features + ['entity_id']],
    train["target"],
    cat_features=[(no_id_features + ['entity_id']).index(c) for c in (no_id_cat_features + ['entity_id'])],
)

CatBoostClassifier(allow_writing_files=False, depth=7, eval_metric='AUC', iterations=2000, l2_leaf_reg=6, learning_rate=0.01, loss_function='Logloss', random_seed=42, verbose=False)

In [73]:
no_entity_model.fit(
    train[no_id_features + ['group_id']],
    train["target"],
    cat_features=[(no_id_features + ['group_id']).index(c) for c in (no_id_cat_features + ['group_id'])],
)

CatBoostClassifier(allow_writing_files=False, depth=7, eval_metric='AUC', iterations=2000, l2_leaf_reg=6, learning_rate=0.01, loss_function='Logloss', random_seed=42, verbose=False)

## inference and result saving for submission

In [74]:
test['is_no_entity'] = test.entity_id.map(lambda x: x in new_entity_ids)
test['is_no_group'] = test.group_id.map(lambda x: x in new_group_ids)

test['cold_start'] = ( test.entity_id.map(lambda x: x in new_entity_ids) &
                       test.group_id.map(lambda x: x in new_group_ids) )

test.cold_start.value_counts(dropna=False)

cold_start
False    61900
True      1370
Name: count, dtype: int64

In [75]:
cold_start_prediction = cold_start_model.predict_proba(test[no_id_features])[:, 1]
deep_prediction = deep_model.predict_proba(test[features])[:, 1]
no_entity_prediction = no_entity_model.predict_proba(test[no_id_features + ['group_id']])[:, 1]
no_group_prediction = no_group_model.predict_proba(test[no_id_features + ['entity_id']])[:, 1]

test['cold_start_prediction'] = cold_start_prediction
test['deep_prediction'] = deep_prediction
test['no_entity_prediction'] = no_entity_prediction
test['no_group_prediction'] = no_group_prediction

In [76]:

def select_model_fn(df):
    cold_start_pred = df['cold_start_prediction']
    deep_pred = df['deep_prediction']
    no_entity_pred = df['no_entity_prediction']
    no_group_pred = df['no_group_prediction']
    cold_start_case = df['cold_start']
    no_entity_case = df['is_no_entity']
    no_group_case = df['is_no_group']

    if cold_start_case:
        return cold_start_pred
    else:
        if no_entity_case:
            return no_group_pred
        elif no_group_case:
            return no_entity_pred
        else:
            return deep_pred

test['prediction'] = test.apply(select_model_fn, axis=1)

test[['row_id','prediction']].to_csv("../results/results_submission_file.csv", index=False)

test.prediction.describe()

count    6.327000e+04
mean     6.114553e-03
std      2.787937e-02
min      1.469774e-07
25%      4.470485e-06
50%      3.999601e-05
75%      1.317559e-03
max      9.760868e-01
Name: prediction, dtype: float64

In [77]:
len(test)

63270

# <<< the END >>>

In [78]:
stop_run = datetime.now()
print('TOTAL RUN TIME WAS', round((stop_run-start_run).total_seconds()), 'seconds')

TOTAL RUN TIME WAS 2492 seconds
